In [ ]:
!pip install requests beautifulsoup4 firebase

import requests

from bs4 import BeautifulSoup

import re

import matplotlib.pyplot as plt

from firebase import firebase

import firebase_admin
from firebase_admin import db

In [ ]:
def fetch_page(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return BeautifulSoup(response.text, "html.parser")
        return None
    except:
        return None


def index_words(soup):
    index = {}
    words = re.findall(r'\w+', soup.get_text())
    for word in words:
        word = word.lower()
        index[word] = index.get(word, 0) + 1
    return index


def remove_stop_words(index):
    stop_words = {'a','an','the','and','or','in','on','at'}
    for w in list(index.keys()):
        if w in stop_words:
            del index[w]
    return index


def build_custom_index(index, keywords, url):
    result = {}
    for word in keywords:
        result[word] = {
            "count": index.get(word, 0),
            "url": url
        }
    return result

In [ ]:
url = "https://en.wikipedia.org/wiki/Plant_disease"

keywords = [
    "disease",
    "plant",
    "fungus",
    "bacteria",
    "virus",
    "infection",
    "leaf",
    "symptom",
    "treatment",
    "pathogen"
]

In [ ]:
soup = fetch_page(url)

index = index_words(soup)
index = remove_stop_words(index)

custom_index = build_custom_index(index, keywords, url)

print(custom_index)

In [ ]:
firebase_url = "https://plant-disaese-default-rtdb.firebaseio.com/"

db = firebase.FirebaseApplication(firebase_url, None)

db.put("/plant_disease_index", "index", index)

In [ ]:
from firebase import firebase

firebase = firebase.FirebaseApplication('https://plant-disaese-default-rtdb.firebaseio.com/', None)

firebase.put('', 'plant_index', custom_index)

In [ ]:
data = firebase.get('/plant_index', None)

words = []
counts = []

if data:
    for word, info in data.items():
        words.append(word)
        counts.append(info["count"])

    plt.bar(words, counts)
    plt.xticks(rotation=45)
    plt.title("Plant Disease Words Frequency")
    plt.show()

else:
    print("No data found in Firebase")